# Prueba rápida: modelo open de Hugging Face

Notebook mínima para verificar que el entorno puede **descargar y ejecutar** un modelo abierto de Hugging Face en local (CPU, sin GPU).

Modelos usados (abiertos, chicos, sin necesidad de token):
- **`distilgpt2`** — generación de texto (decoder-only).
- **`distilbert-base-uncased`** — relleno de máscara (encoder-only).

La primera ejecución descarga los modelos (unos cientos de MB) y los deja cacheados.

## Instalación de dependencias

En local ya están en `requirements.txt`. En Colab, descomentá y corré la celda siguiente.

In [1]:
# !pip install -q transformers torch

## Imports y chequeo del entorno

In [2]:
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import pipeline, set_seed

print('torch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else -1  # -1 = CPU
set_seed(42)

torch: 2.13.0+cpu
CUDA disponible: False


## 1. Generación de texto con `distilgpt2`

Cargamos un pipeline de `text-generation` y le pedimos que continúe un prompt.

In [3]:
generator = pipeline('text-generation', model='distilgpt2', device=DEVICE)

prompt = 'Machine learning is'
salidas = generator(
    prompt,
    max_new_tokens=40,
    num_return_sequences=2,
    do_sample=True,
    top_k=50,
)

for i, s in enumerate(salidas, 1):
    print(f'--- Generación {i} ---')
    print(s['generated_text'])
    print()

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'num_return_sequences', 'max_new_tokens', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- Generación 1 ---
Machine learning is more complex than in the past. In fact we can learn from our mistakes.

--- Generación 2 ---
Machine learning is just a new tool for learning how to teach people how to make money. In this video, we'll show how to teach you how to teach you how to teach people how to make money.




## 2. Relleno de máscara con `distilbert-base-uncased`

El modelo predice la palabra que falta donde está el token `[MASK]`.

In [4]:
unmasker = pipeline('fill-mask', model='distilbert-base-uncased', device=DEVICE)

frase = 'The capital of France is [MASK].'
predicciones = unmasker(frase, top_k=5)

print(f'Frase: {frase}\n')
for p in predicciones:
    print(f"  {p['token_str']:12s} score={p['score']:.4f}")

Frase: The capital of France is [MASK].

  marseille    score=0.1427
  nantes       score=0.0902
  toulouse     score=0.0881
  paris        score=0.0862
  lyon         score=0.0772


## 3. Ejercicio

Cambiá el `prompt` de la Sección 1 y la `frase` de la Sección 2 por texto propio y volvé a correr las celdas. Probá también un modelo distinto (por ejemplo `gpt2` en vez de `distilgpt2`) y compará las salidas.

Si todo corrió sin errores, el entorno está listo para trabajar con modelos de Hugging Face. ✅